# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Christofides (aproximado con garantía 1,5)

<div style="border-left:5px solid #4C72B0;color:#000;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

Es el algoritmo **aproximado con garantía** del viajante: no siempre encuentra el óptimo, pero **asegura que el recorrido no cuesta más de 1,5 veces el óptimo** (Christofides, 1976). Se basa en construir primero una estructura barata que conecta todas las ciudades y después convertirla en un recorrido que pasa una vez por cada una.

**Idea, en 5 pasos.**

1. **Árbol de expansión mínimo (MST).** Se unen todas las ciudades con el menor coste total y sin ciclos. Su coste es una **cota inferior** del óptimo: si a un ciclo óptimo se le quita una arista queda un árbol, y ese árbol no puede costar menos que el mínimo.
2. **Ciudades de grado impar.** En el árbol, algunas ciudades tienen un número impar de aristas (siempre hay un número par de ellas).
3. **Emparejamiento perfecto de coste mínimo** entre esas ciudades: se juntan por parejas con el menor coste posible. Su coste es, como mucho, la mitad del óptimo (de ahí sale el 1,5).
4. **Circuito euleriano.** Árbol más emparejamiento forman un multigrafo en el que **todas las ciudades tienen grado par**, y entonces existe un circuito que recorre **cada arista exactamente una vez**.
5. **Atajos.** El circuito repite ciudades. Se recorre en orden y se **salta cada ciudad ya visitada**. Gracias a la **desigualdad triangular** (ir directo de `x` a `z` nunca cuesta más que pasar por `y`), los atajos no pueden aumentar el coste.

**Por qué 1,5.** `coste ≤ MST + emparejamiento ≤ óptimo + ½·óptimo`.

**Pseudocódigo.**

<pre style="background:#fff;border:1px solid #ccd;padding:10px 14px;border-radius:4px;font-size:13px;">
CHRISTOFIDES(K, inicio):                      # K NO dirigido
    T ← árbol de expansión mínimo de K
    I ← ciudades de grado impar en T
    M ← emparejamiento perfecto de coste mínimo entre las ciudades de I
    G ← T + M                                     # multigrafo con todos los grados pares
    circuito ← circuito euleriano de G que empieza en inicio
    ruta ← circuito sin repetir ciudades (se salta cada ciudad ya visitada)
    devolver (ruta, coste del ciclo)
</pre>

**Implementación.** Usa `NetworkX` para el MST (`nx.minimum_spanning_tree`), el emparejamiento de coste mínimo (`nx.min_weight_matching`, algoritmo *blossom* de Edmonds) y el circuito euleriano (`nx.eulerian_circuit`), pero **el algoritmo completo es el de arriba**, escrito aquí paso a paso. Al final se compara con la versión que trae NetworkX (`nx.approximation.christofides`) como comprobación.

**Complejidad.** **Tiempo: O(n³)** — el MST cuesta `O(n² log n)` en un grafo completo, el emparejamiento *blossom* sobre las ciudades impares `O(n³)` en el peor caso y el circuito euleriano `O(n)`; domina el emparejamiento. **Memoria: O(n²)**, por el grafo completo y el de las ciudades impares.

**Propiedades.**

- **Tipo:** **aproximado con garantía** (heurístico constructivo).
- **¿Siempre encuentra el óptimo?** **No**, pero está **a menos de un 50 %** de él. En la práctica suele quedar mucho más cerca (se mide abajo).
- **¿Determinista?** **Sí** en lo que depende de los datos; los empates en el MST o el emparejamiento los resuelve la biblioteca de forma repetible con la misma entrada.
- **¿Necesita simetría?** **Sí, y desigualdad triangular.** Solo funciona con el grafo **no dirigido** (`general="grafo_simple_no_dirigido"`); con un `nx.DiGraph` la función lanza un `ValueError` claro. Las distancias de carretera son caminos mínimos, así que **cumplen** la desigualdad triangular y la garantía de 1,5 se mantiene.
- **¿Hasta cuántas ciudades?** **Todas las del trabajo**: con 75 tarda milisegundos y llega a cientos sin problema.
- **Uso en el trabajo:** es la única heurística con **garantía demostrada** de la lista; sirve de contraste con las que mejoran la ruta sin prometer nada (2-opt, recocido, genético).

</div>

**💻 CÓDIGO**

La función comprueba primero que el grafo es no dirigido, y después sigue los 5 pasos de la teoría. El emparejamiento se calcula sobre un grafo auxiliar `H` formado solo por las ciudades de grado impar, completo y con los mismos pesos que `K`. Al construir la ruta se conserva el primer paso por cada ciudad del circuito euleriano, que sale de la ciudad de inicio.

In [8]:
def christofides_tsp(K, inicio=None):
    # Christofides para el problema del viajante. Contrato: K es el grafo de ciudades (completo,
    # K[a][b]["peso"]) y devuelve (ruta, coste), con la ruta completa (cada ciudad una vez, el viajante
    # vuelve después a la primera) y el coste del ciclo incluida la vuelta. SOLO sirve para grafos
    # NO dirigidos (necesita distancias simétricas) y con desigualdad triangular, que las distancias
    # por carretera cumplen porque son caminos mínimos. En ese caso, el coste es como mucho 1,5 veces
    # el óptimo.

    if K.is_directed():
        raise ValueError("Christofides necesita un grafo no dirigido (distancias simétricas): "
                         "usa general='grafo_simple_no_dirigido'")
    ciudades = list(K.nodes)
    n = len(ciudades)

    # Casos triviales y comprobaciones
    if n == 0:
        return [], 0.0
    if inicio is None:
        inicio = ciudades[0]
    elif inicio not in K:
        raise KeyError(f"La ciudad de inicio {inicio!r} no está en el grafo")
    if n == 1:
        return [inicio], 0.0

    # 1. Árbol de expansión mínimo (MST): une todas las ciudades con el menor coste total, sin ciclos
    T = nx.minimum_spanning_tree(K, weight="peso")

    # 2. Ciudades de grado impar en el árbol (siempre hay un número par de ellas)
    impares = [v for v, grado in T.degree if grado % 2 == 1]

    # 3. Emparejamiento perfecto de coste mínimo entre ellas: se unen por parejas con el menor coste
    H = nx.Graph()
    for u, v in itertools.combinations(impares, 2):
        H.add_edge(u, v, weight=K[u][v]["peso"])
    pareja = nx.min_weight_matching(H)

    # 4. Árbol + emparejamiento = multigrafo en el que todas las ciudades tienen grado par,
    #    luego existe un circuito euleriano (pasa una vez por cada arista)
    M = nx.MultiGraph(T)
    M.add_edges_from(pareja)
    circuito = [u for u, v in nx.eulerian_circuit(M, source=inicio)]

    # 5. Atajos: se recorre el circuito saltándose las ciudades repetidas. Por la desigualdad
    #    triangular, ir directo nunca cuesta más que dar el rodeo, así que el coste no sube.
    ruta, vistas = [], set()
    for u in circuito:
        if u not in vistas:
            vistas.add(u)
            ruta.append(u)
    coste = sum(K[ruta[i]][ruta[(i + 1) % n]]["peso"] for i in range(n))
    return ruta, float(coste)

**Prueba rápida.** Cuatro comprobaciones:

1. **Versión pequeña real (no dirigida):** las 6 ciudades de Galicia empezando en Vigo, con `validar_ruta` y `medir`.
2. **Referencia exacta:** el mejor de los `6! = 720` ciclos, y el cociente `Christofides / óptimo`, que debe ser **≤ 1,5**.
3. **Grafo dirigido:** debe rechazarlo con un error claro.
4. **Comparación con NetworkX:** `nx.approximation.christofides` implementa el mismo algoritmo.

In [9]:
# ── 1. Versión pequeña real: Galicia (6 ciudades), no dirigido, empezando en Vigo ──
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_simple_no_dirigido")
ruta, coste = christofides_tsp(K, inicio="Vigo")
validar_ruta(K, ruta)                               # comprueba que la ruta visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(christofides_tsp, K, memoria=True, inicio="Vigo"))

# ── 2. Referencia: el mejor de los 6! = 720 ciclos ──
optimo = min(coste_ruta(K, list(p)) for p in itertools.permutations(K.nodes))
print(f"\nóptimo: {optimo:.3f} km | Christofides: {coste:.3f} km | cociente: {coste / optimo:.4f} (garantía: ≤ 1,5)")

# ── 3. Un grafo dirigido debe rechazarse ──
Kdir = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
try:
    christofides_tsp(Kdir)
except ValueError as error:
    print("\nValueError:", error)

# ── 4. Comparación con la implementación de NetworkX ──
ruta_nx = nx.approximation.christofides(K, weight="peso")[:-1]     # devuelve el ciclo repitiendo la primera ciudad
print(f"\nNetworkX: {coste_ruta(K, ruta_nx):.3f} km | la nuestra: {coste:.3f} km")

Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Ourense | coste: 456.9 km
{'algoritmo': 'christofides_tsp', 'ciudades': 6, 'coste': 456.928, 'unidad': 'km', 'segundos': 0.0004, 'memoria_mb': 0.04}

óptimo: 456.928 km | Christofides: 456.928 km | cociente: 1.0000 (garantía: ≤ 1,5)

ValueError: Christofides necesita un grafo no dirigido (distancias simétricas): usa general='grafo_simple_no_dirigido'

NetworkX: 484.546 km | la nuestra: 456.928 km


**Garantía de 1,5 en instancias mayores.** Sobre subgrafos de Andalucía (de 5 a 9 ciudades) y de las capitales de España (de 8 a 11) se calcula el óptimo enumerando todos los ciclos y se comprueba que el cociente **nunca supera 1,5**. Esta celda falla (`assert`) si la garantía se rompiera.

In [10]:
# ── Cociente Christofides / óptimo, con la garantía comprobada ──
KAnd = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_simple_no_dirigido")
KEsp = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_simple_no_dirigido")
print(f"{'instancia':>18} | {'óptimo':>9} | {'Christofides':>12} | {'cociente':>8}")
for nombre, Kbase, tamanos in (("Andalucía", KAnd, range(5, 10)), ("capitales España", KEsp, range(8, 12))):
    for n in tamanos:
        Kn = Kbase.subgraph(list(Kbase.nodes)[:n]).copy()
        primera = list(Kn.nodes)[0]
        optimo = min(coste_ruta(Kn, [primera] + list(p)) for p in itertools.permutations(list(Kn.nodes)[1:]))
        _, c = christofides_tsp(Kn)
        assert c <= 1.5 * optimo + 1e-9, "se ha roto la garantía de 1,5"
        print(f"{nombre + ' ' + str(n):>18} | {optimo:>9.1f} | {c:>12.1f} | {c / optimo:>8.4f}")

         instancia |    óptimo | Christofides | cociente
       Andalucía 5 |     853.7 |        967.4 |   1.1333
       Andalucía 6 |    1034.6 |       1149.3 |   1.1108
       Andalucía 7 |    1068.3 |       1069.2 |   1.0009
       Andalucía 8 |    1213.4 |       1214.3 |   1.0007


       Andalucía 9 |    1214.8 |       1214.8 |   1.0000


capitales España 8 |    2272.5 |       2527.5 |   1.1122


capitales España 9 |    2642.1 |       3101.1 |   1.1737


capitales España 10 |    2642.7 |       3101.7 |   1.1737


capitales España 11 |    2720.9 |       2960.7 |   1.0882


**Cota inferior y escalabilidad.** Con 47 o 75 ciudades no se puede enumerar el óptimo, pero el coste del **MST es una cota inferior** (`MST ≤ óptimo`), así que `Christofides / MST` es una **cota superior del cociente real** `Christofides / óptimo`. Se mide en todas las versiones **no dirigidas** de España.

In [11]:
# ── Escalabilidad y cota: todas las versiones no dirigidas de España ──
print(f"{'aristas':>21} | {'ciudades':>8} | {'MST (cota)':>10} | {'Christofides':>12} | {'cociente máx.':>13} | {'tiempo':>8}")
for aristas in ("completo", "autovias_principales", "sin_autovias", "solo_autovias"):
    for nodos in ("capitales", "capitales_y_100k"):
        Kv = cargar_version(aristas=aristas, nodos=nodos, ambito="espana", peso="km", general="grafo_simple_no_dirigido")
        cota = nx.minimum_spanning_tree(Kv, weight="peso").size(weight="peso")     # coste del MST
        r = medir(christofides_tsp, Kv)
        print(f"{aristas:>21} | {r['ciudades']:>8} | {cota:>10.1f} | {r['coste']:>12.1f} | {r['coste'] / cota:>13.3f} | {r['segundos']:>7.3f}s")

              aristas | ciudades | MST (cota) | Christofides | cociente máx. |   tiempo
             completo |       47 |     4884.9 |       6892.7 |         1.411 |   0.010s
             completo |       75 |     5232.8 |       6898.1 |         1.318 |   0.022s
 autovias_principales |       47 |     5055.4 |       7130.9 |         1.411 |   0.010s
 autovias_principales |       75 |     5388.1 |       7062.4 |         1.311 |   0.021s
         sin_autovias |       47 |     6266.6 |       8500.5 |         1.356 |   0.013s
         sin_autovias |       75 |     6867.8 |       9499.9 |         1.383 |   0.049s
        solo_autovias |       44 |     5379.5 |       7721.4 |         1.435 |   0.009s


        solo_autovias |       72 |     5774.0 |       8015.3 |         1.388 |   0.032s


<div style="border-left:5px solid #55A868;color:#000;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

**Prueba rápida (Galicia, 6 ciudades, no dirigido, desde Vigo).** Ruta devuelta:

> **Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Ourense → Vigo = 456,9 km**

que coincide con el **óptimo** de esta versión (456,928 km; cociente 1,0000). Con el grafo **dirigido** la función se niega con un `ValueError` que indica qué versión usar. `nx.approximation.christofides` de NetworkX devuelve un recorrido válido de 484,6 km: ambos respetan la garantía, pero el circuito euleriano admite varios recorridos y los atajos producen rutas distintas.

**Garantía de 1,5 comprobada** (9 instancias de Andalucía y de las capitales de España, de 5 a 11 ciudades, óptimo por enumeración): el cociente va de **1,0000 a 1,1639**, siempre por debajo de 1,5 (el `assert` de la celda lo verifica). En 4 de las 9 instancias da el óptimo o casi (≤ 1,001); el peor caso, 1,16 (9 y 10 capitales).

**Escalabilidad y cota** (versiones no dirigidas de España, km). Como `MST ≤ óptimo`, el cociente `Christofides / MST` acota por arriba el error real:

| Aristas | Ciudades | MST (cota inferior) | Christofides | Cociente máx. | Tiempo |
|---|---:|---:|---:|---:|---:|
| completo | 47 | 4.884,9 | 6.892,7 | 1,411 | 0,02 s |
| completo | 75 | 5.232,8 | 6.898,1 | 1,318 | 0,02 s |
| autovías principales | 47 | 5.055,4 | 7.130,9 | 1,411 | 0,01 s |
| sin autovías | 75 | 6.867,8 | 9.499,9 | 1,383 | 0,05 s |
| solo autovías | 72 | 5.774,0 | 8.015,3 | 1,388 | 0,03 s |

En todas las versiones el cociente máximo queda entre **1,31 y 1,44**, dentro de la garantía; el error real es menor (la cota usa el MST, que queda por debajo del óptimo). Para las 47 capitales no dirigidas, el genético de este trabajo encontró una ruta de 5.940,2 km, así que Christofides (6.892,7 km) queda **al menos un 16 % por encima del óptimo**, y el 2-opt (6.267,3 km) le gana en la misma instancia: la garantía es un peor caso, no una predicción. Tarda **centésimas de segundo** y es la única de estas heurísticas con garantía demostrada.

</div>

---